In [1]:
%load_ext autoreload
%autoreload 2
import pandas as pd 
import numpy as np
import glob
import os
import matplotlib.pyplot as plt
import fonctions 
import embeddingsFunctions as ef
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score
import networkx as nx
from node2vec import Node2Vec
from sklearn.model_selection import train_test_split
import umap
import seaborn as sns
from sklearn.cluster import KMeans
from collections import Counter
from matplotlib.lines import Line2D
import pickle
from BaryCentreClassifier import BarycenterClassifier
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

In [2]:
# Initialisation de la liste avec les noms des fichiers (sans l'extension .csv)
# unique cellule a changer , on donne les fichiers a qui on veut appliquer le pipeline
# les fichiers doivent être au format csv
# dans la liste on met les noms des fichier sans les .csv
list_files = [
    "part_01", "part_02", "part_03", "part_04", "part_05", 
    "part_06", "part_07", "part_08", "part_09", "part_10"
]

In [ ]:
# Cellule qui lit les fichier puis créer des graphes et les sauvgardes sous plusieurs formats
# Initialisation du dictionnaire vide avant la boucle
dictionnaire_graphes = {}

# Création et sauvegarde des graphes en parcourant la liste
for fichier in list_files:
    # Lecture du fichier CSV (on ajoute l'extension .csv ici)
    df = pd.read_csv(f"{fichier}.csv")
    
    # Création du graphe
    G = nx.from_pandas_edgelist(
        df,
        source='user_id',
        target='original_author',
        edge_attr='nb_retweeted'
    )
    
    # Ajout du graphe dans le dictionnaire avec le nom de base comme clé
    dictionnaire_graphes[fichier] = G
    
    # Sauvegarde au format pickle
    # pour pouvoir sauvegrader le graphe et le relire d'un fichier sans executer toute la cellule et le recuperer du dict 
    with open(f"graph_{fichier}.gpickle", "wb") as f:
        pickle.dump(G, f)
    
    # Sauvegarde au format GEXF compatible avec GEPHI
    nx.write_gexf(G, f"graph_{fichier}.gexf")
    
    # Message de confirmation
    print(f"graph_{fichier} sauvegardé et ajouté au dictionnaire")

# Pour vérifier que le dictionnaire contient bien tes graphes :
print("\nClés du dictionnaire :", dictionnaire_graphes.keys())

graph_part_01 sauvegardé et ajouté au dictionnaire
graph_part_02 sauvegardé et ajouté au dictionnaire
graph_part_03 sauvegardé et ajouté au dictionnaire
graph_part_04 sauvegardé et ajouté au dictionnaire
graph_part_05 sauvegardé et ajouté au dictionnaire
graph_part_06 sauvegardé et ajouté au dictionnaire
graph_part_07 sauvegardé et ajouté au dictionnaire


In [ ]:
#Cellule pas importante: pour visualiser un graphe si cela est voulu 

# Pour visualiser un graphe spécifique
nom_graphe = 'part_01'  # À remplacer par une clé existante
G = dictionnaire_graphes[nom_graphe]

# Visualisation basique
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, k=1, iterations=50)
nx.draw(G, pos, with_labels=True, node_size=500, node_color='lightblue', 
        font_size=8, font_weight='bold')
plt.title(f"Graphe des retweets - {nom_graphe}")
plt.show()

# Ou avec les poids des arêtes (nb_retweeted)
plt.figure(figsize=(12, 8))
pos = nx.spring_layout(G, k=1, iterations=50)
edges = G.edges()
weights = [G[u][v]['nb_retweeted'] for u, v in edges]

nx.draw(G, pos, with_labels=True, node_size=500, node_color='lightblue',
        font_size=8, font_weight='bold', width=[w/10 for w in weights])
plt.title(f"Graphe des retweets (épaisseur = nb_retweeted) - {nom_graphe}")
plt.show()

In [ ]:
# Cellule qui lit les fichier des graph puis applique Node2Vec pour tous les graphes de la liste et sauvgarde les resultats
# on execute cette cellule une seule fois car elle prend bcp de temps
# si on a pas deja les fichiers embeddings32_{fichier}.csv et si on les a pas la peine de l'executer

# Parcours de tous les fichiers dans list_files
for fichier in list_files:
    print(f"\nTraitement de Node2Vec : {fichier}")
    
    # Rechargement du graphe déjà créé
    with open(f"graph_{fichier}.gpickle", "rb") as f:
        G = pickle.load(f)
    
    print(f"Nombre de nœuds : {G.number_of_nodes()}")
    print(f"Nombre d'arêtes : {G.number_of_edges()}")
    
    # Application de node2vec pour les embeddings 32 dimensions
    ef.embedding32dimensions(G)
    
    # Renommage du fichier de sortie
    if os.path.exists('embeddings32dimensions.csv'):
        os.rename('embeddings32dimensions.csv', f'embeddings32_{fichier}.csv')
        print(f"Embeddings sauvegardés dans embeddings32_{fichier}.csv")
    
    print(f"Terminé pour {fichier}\n")
    
    # Libération de la mémoire
    del G

In [ ]:
# Cellule qui charge le fichier des nodes pour le suppervisé 
# Chargement du fichier des utilisateurs (Nodes)
df_nodes = pd.read_csv('nodes.csv', sep=';', dtype=str)
print(f"Nombre de nœuds : {len(df_nodes)}")
print(f"Colonnes : {df_nodes.columns.tolist()}")

# Les IDs sont déjà dans la première colonne
ids_valides = set(df_nodes['Id'])
print(f"Nombre d'IDs uniques : {len(ids_valides)}")

In [ ]:
# Cellule qui charge les fichier des embedding et qui les enregistre dans un dict 
# Dictionnaire pour stocker les embeddings
dictionnaire_embeddings = {}

# Parcours de tous les fichiers dans list_files
for fichier in list_files:
    dictionnaire_embeddings[fichier] = pd.read_csv(f'embeddings32_{fichier}.csv')

# Vérification
print("\nClés du dictionnaire :", dictionnaire_embeddings.keys())

In [ ]:
# Cellule qui s'occupe des filtrations et des splites
# On filtre (on laisse que les id present dans le fichier des node) et on splite les utilisateurs
# Split train/test
splits = {}
for fichier, df in dictionnaire_embeddings.items():
    col_id = df.columns[0]
    df[col_id] = df[col_id].astype(str)
    df_filtre = df[df[col_id].isin(ids_valides)]
    #ici on peut mettre dans le test_size la taille qu'on veut
    train, test = train_test_split(df_filtre, test_size=0.8, random_state=42)
    splits[fichier] = (train, test)
    print(f"{fichier}: {len(train)} train, {len(test)} test")

In [ ]:
# Celulle nessaissaire pour extraire les labels utilisées dans le supervisé de umap
# Création du dictionnaire de mapping ID -> label
mapping_labels = {}

for index, row in df_nodes.iterrows():
    id_user = str(row['Id'])  # L'ID utilisateur
    label = row['modularity_class']  # La classe (0 ou 4)
    mapping_labels[id_user] = label

print(f"Nombre de labels chargés : {len(mapping_labels)}")
print(f"Exemple : {list(mapping_labels.items())[:5]}")

In [ ]:
# Cellule qui fait le umap supervisé 
# Dictionnaire pour stocker les projections UMAP supervisées
projections_supervisees = {}

for fichier in list_files:
    print(f"\n--- Traitement de {fichier} ---")
    
    # Récupérer les splits déjà préparés
    train_df, test_df = splits[fichier]
    
    if len(train_df) == 0 or len(test_df) == 0:
        print(f"   Split vide pour {fichier}, ignoré")
        continue
    
    # Extraction des features (X) : toutes les colonnes sauf la 1ère (ID)
    X_train = np.array([row[1:] for row in train_df.values], dtype=np.float32)
    X_test = np.array([row[1:] for row in test_df.values], dtype=np.float32)
    
    # Extraction des labels (y) et conversion en entier
    y_train = [int(mapping_labels[str(row[0])]) for row in train_df.values]
    y_test = [int(mapping_labels[str(row[0])]) for row in test_df.values]
    
    print(f"  Train: {len(X_train)} points, Test: {len(X_test)} points")
    print(f"  Dimension embeddings: {X_train.shape[1]}D")
    print(f"  Labels train: 0={y_train.count(0)}, 4={y_train.count(4)}")
    
    # UMAP supervisé
    reducer = umap.UMAP(
        n_neighbors=15, 
        min_dist=0.1, 
        n_components=2, 
        random_state=42
    )
    
    # Fit sur les données d'entraînement AVEC les labels (maintenant numériques)
    train_2d = reducer.fit_transform(X_train, y=y_train)
    
    # Transform sur les données de test (sans labels, même transformation)
    test_2d = reducer.transform(X_test)
    
    # Stockage
    projections_supervisees[fichier] = {
        'train': train_2d,
        'test': test_2d,
        'y_train': y_train,
        'y_test': y_test
    }
    
    print(f"  UMAP supervisé terminé")

print("\n" + "="*50)
print(f"Projections supervisées prêtes pour {len(projections_supervisees)} fichiers")

In [ ]:
# fonction qui plot les resultat de umap 
fonctions.plot_umap_projections(projections_supervisees)

In [ ]:
# Cellule qui calcul la presision de umap avec la methode des barrycentres
# Dictionnaire pour stocker les résultats des classifieurs par fichier
resultats_classifieurs = {}

for fichier, data in projections_supervisees.items():
    print(f"\n{'='*50}")
    print(f"Traitement de {fichier}")
    print(f"{'='*50}")
    
    # Récupération des données train et test
    X_train = data['train']      # coordonnées UMAP 2D du train
    y_train = data['y_train']    # labels du train (0 et 4)
    X_test = data['test']        # coordonnées UMAP 2D du test
    y_test = data['y_test']      # labels du test
    
    print(f"Train: {len(X_train)} points")
    print(f"Test: {len(X_test)} points")
    
    # Création et entraînement du classifieur
    clf = BarycenterClassifier(label_positive=4, label_negative=0, target=(2, 2))
    clf.fit(X_train, y_train)
    
    # Évaluation sur le train
    train_accuracy = clf.score(X_train, y_train)
    print(f"\n Accuracy sur TRAIN: {train_accuracy:.4f} ({train_accuracy*100:.2f}%)")
    
    # Évaluation sur le test
    test_accuracy = clf.score(X_test, y_test)
    print(f" Accuracy sur TEST: {test_accuracy:.4f} ({test_accuracy*100:.2f}%)")
    
    # Stockage du classifieur et des métriques
    resultats_classifieurs[fichier] = {
        'classifier': clf,
        'train_accuracy': train_accuracy,
        'test_accuracy': test_accuracy,
        'X_train': X_train,
        'y_train': y_train,
        'X_test': X_test,
        'y_test': y_test
    }
    
    # Affichage de la matrice de confusion pour le test
    print(f"\n Matrice de confusion sur TEST:")
    clf.plot_confusion_matrix(X_test, y_test, title=f"{fichier} - Test")
    
    # Affichage du plot des points transformés pour le test
    clf.plot(X_test, y_test, title=f"{fichier} - Projection transformée (Test)")

# Résumé final
print("\n" + "="*60)
print("RÉSUMÉ DES ACCURACIES")
print("="*60)

for fichier, res in resultats_classifieurs.items():
    print(f"{fichier}: Train = {res['train_accuracy']:.4f} | Test = {res['test_accuracy']:.4f}")

# Calcul de l'accuracy moyenne sur le test
avg_test_accuracy = np.mean([res['test_accuracy'] for res in resultats_classifieurs.values()])
print(f"\n Accuracy moyenne sur TEST (tous fichiers): {avg_test_accuracy:.4f} ({avg_test_accuracy*100:.2f}%)")

In [ ]:
# Cellule qui calcul la presision de umap avec la methode de KNN
resultats_knn = {}

for fichier, data in projections_supervisees.items():
    print("\n" + "="*50)
    print(f"KNN - {fichier}")
    print("="*50)
    
    X_train, y_train = data['train'], data['y_train']
    X_test, y_test = data['test'], data['y_test']
    
    print(f"Train: {len(X_train)} points | Test: {len(X_test)} points")
    
    # KNN
    knn = KNeighborsClassifier(n_neighbors=5)
    knn.fit(X_train, y_train)
    y_pred = knn.predict(X_test)
    
    # Métriques
    acc = accuracy_score(y_test, y_pred)
    resultats_knn[fichier] = {
        'accuracy': acc,
        'knn_model': knn
    }
    
    print(f"\nAccuracy: {acc:.4f}")
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred, target_names=['Pro-climat (0)', 'Sceptique (4)']))

# Moyenne
avg_acc = np.mean([res['accuracy'] for res in resultats_knn.values()])
print("\n" + "="*50)
print(f"ACCURACY MOYENNE SUR {len(resultats_knn)} FICHIERS: {avg_acc:.4f}")

In [ ]:
# Celulle qui affichie les matrice de confusion pour le KNN
labels = ['Pro-climat (0)', 'Sceptique (4)']

# Création de la grille 2x5
fig, axes = plt.subplots(2, 5, figsize=(20, 8))
axes = axes.flatten()

for idx, fichier in enumerate(list_files):
    ax = axes[idx]
    
    # Récupération des données
    data = projections_supervisees[fichier]
    
    # Récupération du modèle KNN depuis resultats_knn
    knn = resultats_knn[fichier]['knn_model']
    
    # Prédiction
    y_test = data['y_test']
    y_pred = knn.predict(data['test'])
    
    # Matrice de confusion
    cm = confusion_matrix(y_test, y_pred)
    acc = resultats_knn[fichier]['accuracy']
    
    # Affichage
    disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=labels)
    disp.plot(ax=ax, cmap='Blues', colorbar=False, values_format='d')
    ax.set_title(f"{fichier}\nAcc: {acc:.3f}", fontsize=10)

plt.suptitle("Matrices de Confusion KNN - UMAP Supervisé (10 parties)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()